# Classical ML: phase plan

Goal: train logistic regression, random forest, k-NN and SVM on MNIST, save them, and
compare them fairly with the CNN. Each phase ends with something that works, so you can stop
and commit after any of them.

---

## Phase 1: Shared data loading (`src/data.py`)

Both the CNN and the classical models depend on this, so it comes first.

- [ ] Use one fixed data path based on the project root, e.g.
      `Path(__file__).resolve().parents[1] / "data"`, so it works from scripts and notebooks alike.
- [ ] Load MNIST train (60k) and test (10k) with torchvision.
- [ ] Split train into **55k train / 5k validation** with a fixed seed (e.g. `42`).
- [ ] A function that returns numpy arrays for the classical models:
  - `X` shape `(N, 784)`, `float32`, scaled to **[0, 1]** (divide by 255)
  - `y` shape `(N,)`, ints 0–9
- [ ] A function that returns DataLoaders for the CNN, using **the same split indices**.
- [ ] Test set is only used in Phase 5.

**Done when:** `from src.data import ...` works in a notebook and the shapes are
`(55000, 784)`, `(5000, 784)` and `(10000, 784)`.

> Tip: `train_data.data` is a `uint8` tensor of shape `(60000, 28, 28)`, and `train_data.targets`
> holds the labels. Reshaping with `.reshape(-1, 784)` is much faster than looping over the dataset.

---

## Phase 2: Model definitions (`src/models/classical.py`)

Build one function that returns a dict of **unfitted** sklearn `Pipeline`s, so scaling and PCA
are saved together with each model.

| Name | Pipeline | Expected accuracy |
|---|---|---|
| `logreg` | `StandardScaler` → `LogisticRegression(max_iter=1000)` | ~92% |
| `random_forest` | `RandomForestClassifier(n_estimators=200, n_jobs=-1)` | ~97% |
| `knn` | `PCA(50)` → `KNeighborsClassifier(n_neighbors=3, n_jobs=-1)` | ~97.5% |
| `svm` | `StandardScaler` → `PCA(50)` → `SVC(kernel="rbf")` | ~98% |
| *(optional)* `hog_svm` | HOG features → `LinearSVC` | ~98% |

- [ ] Set `random_state=42` wherever a model has one.
- [ ] Keep the settings in one place (a dict or constants at the top) so Phase 3 can change them easily.

**Done when:** `get_models()` returns all pipelines and each one can `.fit()` on 1,000 samples
without errors.

---

## Phase 3: Exploration (`notebooks/02_classical.ipynb`)

Try things out here before writing the training script.

- [ ] **PCA check:** plot cumulative explained variance; find how many components keep ~90%.
      Look at a few images reconstructed from 50 components.
- [ ] **Tune on a 10k subset**, scored on the **validation** set (never the test set):
  - k-NN: `k` ∈ {1, 3, 5, 7}
  - SVM: `C` ∈ {1, 5, 10}, `gamma` ∈ {"scale", 0.01}
  - PCA components: {30, 50, 100}
  - Random forest: `n_estimators` ∈ {100, 200, 300}
- [ ] Record **training time** and **prediction time** for each model.
- [ ] Look at a few misclassified digits for each model.
- [ ] Write the chosen settings back into `src/models/classical.py`.

**Done when:** you have the final settings for each model and a short note on why you chose them.

> SVM training time grows faster than linearly with the number of samples. Tune on 10k,
> then train on the full 55k only once.

---

## Phase 4: Training script (`src/train_classical.py`)

- [ ] Load train and validation data from `src/data.py`.
- [ ] For each pipeline: fit on the training split, time it with `time.perf_counter()`,
      and print validation accuracy.
- [ ] Save each one with `joblib.dump(pipe, f"models/{name}.joblib")`.
- [ ] Save training times too (e.g. `reports/train_times.csv`) for the final table.
- [ ] Optional: `--only svm` argument to train a single model.

**Done when:** `python -m src.train_classical` runs from scratch and creates a `.joblib`
file for each model in `models/`.

---

## Phase 5: Comparison with the CNN (`src/evaluate.py`)

Before this phase, the CNN must be saved to `models/cnn.pt` (move the notebook code to
`src/models/cnn.py` + `src/train_cnn.py`).

For every model, on the **same 10k test set**:

- [ ] Test accuracy
- [ ] Prediction time (total for 10k, and per image)
- [ ] File size on disk
- [ ] Training time (from Phase 4)
- [ ] Confusion matrix → `reports/figures/cm_<name>.png`
- [ ] Grid of misclassified examples → `reports/figures/errors_<name>.png`
- [ ] Write everything to `reports/results.csv`

**Done when:** `python -m src.evaluate` produces `results.csv` and the figures, and you can
fill in the README results table.

---

## Phase 6: Write-up (README)

- [ ] Fill in the results table from `results.csv`.
- [ ] **Key findings**, e.g.:
  - How big is the accuracy gap between the CNN and the best classical model?
  - Which digits are confused (4↔9, 3↔5, 7↔2?) and do all models make the same mistakes?
  - Trade-offs: k-NN and random forest files are large; the CNN is small but needs PyTorch.
- [ ] Commit and push.

---

## Things to watch for

- **Scaling:** classical models use [0, 1] pixels plus their own scaler, not the CNN's
  `Normalize(0.1307, 0.3081)`.
- **Model size:** k-NN stores all 55k training rows; a random forest with many trees can be
  hundreds of MB. Check sizes before deploying to Hugging Face.
- **CPU only:** sklearn doesn't use the GPU. Use `n_jobs=-1` where available.
- **Fairness:** every model uses the same training split, tuning uses only the validation set,
  and the test set is used once at the end.
- **Demo:** classical models need input shaped `(1, 784)` scaled to [0, 1];
  `src/preprocess.py` must output that as well as the CNN tensor.
